In [19]:
import os

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

In [20]:
from groq import Groq

MODEL_NAME = "llama-3.3-70b-versatile"

if not api_key:
    raise ValueError("Please set the GROQ_API_KEY environment variable.")

client = Groq(api_key=api_key)

### 1. Non Looping Agent

In [21]:
def call_llm(prompt: str) -> str:
    """Send one request to the language model."""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful customer-support agent.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content.strip()

task = """
Write a customer-support reply for the following situation:

A customer received a damaged laptop.
Order ID: ORD-2026-101

Requirements:
- Apologize to the customer
- Mention the order ID
- Offer a replacement
- Keep the response under 60 words
"""
answer = call_llm(task)

print("Final answer:")
print(answer)

Final answer:
Sorry for the damaged laptop (ORD-2026-101). We'll replace it promptly.


### 2. Looping Agent

In [22]:
MAX_ITERATIONS = 3
ORDER_ID = "ORD-2026-101"

def call_llm(prompt: str) -> str:
    """Generate a customer-support response."""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful customer-support agent. "
                    "Follow every requirement carefully."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content.strip()

def evaluate_answer(answer: str) -> list[str]:
    """
    Check whether the answer satisfies all requirements.

    An empty list means that the answer passed.
    """

    problems = []
    lower_answer = answer.lower()

    # Check for an apology
    apology_words = ["sorry", "apologize", "apologies"]

    if not any(word in lower_answer for word in apology_words):
        problems.append("Add an apology to the customer.")

    # Check the order ID
    if ORDER_ID not in answer:
        problems.append(f"Mention the order ID {ORDER_ID}.")

    # Check whether a replacement was offered
    if "replacement" not in lower_answer:
        problems.append("Clearly offer a replacement.")

    # Check the word limit
    word_count = len(answer.split())

    if word_count > 60:
        problems.append(
            f"Reduce the response to under 60 words. "
            f"Current length: {word_count} words."
        )

    return problems

task = f"""
Write a customer-support reply for the following situation:

A customer received a damaged laptop.
Order ID: {ORDER_ID}

Requirements:
- Apologize to the customer
- Mention the order ID
- Offer a replacement
- Keep the response under 60 words
"""

feedback = "No previous feedback."
previous_answer = ""


for iteration in range(1, MAX_ITERATIONS + 1):

    print(f"\n--- Iteration {iteration} ---")

    prompt = f"""
Original task:
{task}

Previous answer:
{previous_answer or "No previous answer."}

Feedback:
{feedback}

Write an improved final response.
"""

    # Agent performs an action
    answer = call_llm(prompt)

    print("\nGenerated answer:")
    print(answer)

    # System evaluates the action
    problems = evaluate_answer(answer)

    if not problems:
        print("\nEvaluation: PASSED")
        print("\nFinal answer:")
        print(answer)
        break

    print("\nEvaluation: FAILED")

    for problem in problems:
        print(f"- {problem}")

    # Feedback becomes input for the next iteration
    feedback = "\n".join(f"- {problem}" for problem in problems)
    previous_answer = answer

else:
    print("\nMaximum iterations reached.")
    print("Returning the best available answer:")
    print(previous_answer)


--- Iteration 1 ---

Generated answer:
Sorry for the damaged laptop (ORD-2026-101). We'll replace it.

Evaluation: FAILED
- Clearly offer a replacement.

--- Iteration 2 ---

Generated answer:
Sorry for the damaged laptop (ORD-2026-101). We'll provide a replacement laptop.

Evaluation: PASSED

Final answer:
Sorry for the damaged laptop (ORD-2026-101). We'll provide a replacement laptop.
